In [1]:
pip install -U scikit-learn --user

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 61.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from time import time
import numpy as np
import os
from glob import glob
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform
from matplotlib.colors import ListedColormap
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.model_selection import GridSearchCV

np.random.seed(42)
mlp = Pipeline(
    [
        ('standardscaler', StandardScaler()),
        ('mlpclassifier',
         MLPClassifier(max_iter=3000, early_stopping=True)),
    ]
)
parameter_space = {
    'mlpclassifier__hidden_layer_sizes': [(10,10,10), (20,20), (30,10),(40,)],
    'mlpclassifier__solver': [ 'lbfgs'],
    'mlpclassifier__alpha': np.logspace(-1, 1, 3),
    #'learning_rate': ['constant', 'adaptive'],
}


clf = GridSearchCV(mlp, parameter_space, n_jobs=-1, cv=5)

file_path='/data/yangf7/identity/';
file_path_n1='/data/yangf7/identity/night1/';

Xlist = sorted(glob(os.path.join(file_path,'*_FCs.csv')))
ylist = sorted(glob(os.path.join(file_path,'*_ids.csv')))


X_cell = [pd.read_csv(file, header=None).to_numpy() for file in Xlist]
y_cell = [pd.read_csv(file, header=None).to_numpy().ravel() for file in ylist]


Xlist_n1 = sorted(glob(os.path.join(file_path_n1,'*_FCs.csv')))
ylist_n1 = sorted(glob(os.path.join(file_path_n1,'*_ids.csv')))

X_cell_n1 = [pd.read_csv(file, header=None).to_numpy() for file in Xlist_n1]
y_cell_n1 = [pd.read_csv(file, header=None).to_numpy().ravel() for file in ylist_n1]



In [8]:
scores = []
scores_n1 = []
clfs = []
np.random.seed(42)
for X, y in zip(X_cell, y_cell):
    #X_train, X_test, y_train, y_test = train_test_split(
    #    X, y, test_size=0.2, random_state=41
    #)
    clf.fit(X, y)
    print('Best parameters found:\n', clf.best_params_)

    means = clf.cv_results_['mean_test_score']
    stds = clf.cv_results_['std_test_score']
    for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print(f"{mean:.3f} (+/-{std * 2:.03f}) for {params}")
    for xx, yy in zip(X_cell, y_cell):
        score = clf.best_estimator_.score(xx, yy)
        print(score)
        scores.append(score)
    for x1, y1 in zip(X_cell_n1, y_cell_n1):
        score_n1 = clf.best_estimator_.score(x1, y1)
        print(score_n1)
        scores_n1.append(score_n1)
    clfs.append(clf.best_estimator_)
print(np.array(scores).reshape(5,5))


/usr/local/apps/python/py3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best parameters found:
 {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (30, 10), 'mlpclassifier__solver': 'lbfgs'}
0.910 (+/-0.126) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.991 (+/-0.015) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.997 (+/-0.009) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (30, 10), 'mlpclassifier__solver': 'lbfgs'}
0.991 (+/-0.026) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (40,), 'mlpclassifier__solver': 'lbfgs'}
0.958 (+/-0.052) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.988 (+/-0.034) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.979 (+/-0.034) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__h

In [5]:

scores = []
scores_n1 = []
clfs = []

np.random.seed(42)

FIXED_TRAIN_N = 400
BASE_RS = 42  # base random state

for i, (X, y) in enumerate(zip(X_cell, y_cell)):
    n = len(X)

    # Split rule:
    # - if n >= 400: fixed 400 train samples
    # - else: 80/20 split
    if n >= FIXED_TRAIN_N:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            train_size=FIXED_TRAIN_N,
            test_size=n - FIXED_TRAIN_N,
            shuffle=True,
            random_state=BASE_RS + i,
            stratify=y  # set to None if regression / or rare-class issues
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            shuffle=True,
            random_state=BASE_RS + i,
            stratify=None  # set to None if regression / or rare-class issues
        )

    # Fit (GridSearchCV or similar) on TRAIN ONLY
    clf.fit(X_train, y_train)
    print('Best parameters found:\n', clf.best_params_)

    means = clf.cv_results_['mean_test_score']
    stds = clf.cv_results_['std_test_score']
    for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print(f"{mean:.3f} (+/-{std * 2:.03f}) for {params}")

    best_model = clf.best_estimator_
    clfs.append(best_model)

    # Evaluate: (A) test split of the same dataset i
    test_score_i = best_model.score(X_test, y_test)
    print(f"Loop {i}: test score on its own split = {test_score_i:.4f}")

    # Evaluate: (B) your original "cross-cell" evaluation,
    # BUT now it's on the held-out TEST split for each dataset j
    for j, (Xj, yj) in enumerate(zip(X_cell, y_cell)):
        nj = len(Xj)
        if nj >= FIXED_TRAIN_N:
            _, Xj_test, _, yj_test = train_test_split(
                Xj, yj,
                train_size=FIXED_TRAIN_N,
                test_size=nj - FIXED_TRAIN_N,
                shuffle=True,
                random_state=BASE_RS + j,
                stratify=yj
            )
        else:
            _, Xj_test, _, yj_test = train_test_split(
                Xj, yj,
                test_size=0.2,
                shuffle=True,
                random_state=BASE_RS + j,
                stratify=None
            )

        s = best_model.score(Xj_test, yj_test)
        print(f"train i={i} -> test j={j}: {s:.4f}")
        scores.append(s)

    # N1 evaluation (assuming X_cell_n1/y_cell_n1 are separate sets; score on ALL of them as provided)
    for j, (x1, y1) in enumerate(zip(X_cell_n1, y_cell_n1)):
        s_n1 = best_model.score(x1, y1)
        print(f"train i={i} -> N1 set j={j}: {s_n1:.4f}")
        scores_n1.append(s_n1)

print(np.array(scores).reshape(5, 5))

/usr/local/apps/python/py3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Best parameters found:
 {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.981 (+/-0.024) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.992 (+/-0.019) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.989 (+/-0.019) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (30, 10), 'mlpclassifier__solver': 'lbfgs'}
0.992 (+/-0.019) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (40,), 'mlpclassifier__solver': 'lbfgs'}
0.966 (+/-0.044) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.981 (+/-0.024) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.974 (+/-0.018) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__h

/usr/local/apps/python/py3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Best parameters found:
 {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
1.000 (+/-0.000) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
1.000 (+/-0.000) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
1.000 (+/-0.000) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (30, 10), 'mlpclassifier__solver': 'lbfgs'}
1.000 (+/-0.000) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (40,), 'mlpclassifier__solver': 'lbfgs'}
1.000 (+/-0.000) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.997 (+/-0.010) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.997 (+/-0.010) for {'mlpclassifier__alpha': 1.0, 'mlpclassifie

/usr/local/apps/python/py3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Best parameters found:
 {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.970 (+/-0.085) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.995 (+/-0.012) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.992 (+/-0.020) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (30, 10), 'mlpclassifier__solver': 'lbfgs'}
0.988 (+/-0.032) for {'mlpclassifier__alpha': 0.1, 'mlpclassifier__hidden_layer_sizes': (40,), 'mlpclassifier__solver': 'lbfgs'}
0.990 (+/-0.019) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (10, 10, 10), 'mlpclassifier__solver': 'lbfgs'}
0.990 (+/-0.019) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (20, 20), 'mlpclassifier__solver': 'lbfgs'}
0.990 (+/-0.019) for {'mlpclassifier__alpha': 1.0, 'mlpclassifier__h

In [ ]:
np.savetxt(os.path.join(file_path, '400score.txt'),np.array(scores).reshape(5,5))
np.savetxt(os.path.join(file_path, '400score_n1.txt'),np.array(scores_n1).reshape(5,5))

In [1]:
import shap

ModuleNotFoundError: No module named 'shap'

In [2]:
[[1.         0.92479109 0.79581152 0.79585327 0.66067864]
 [0.99697885 1.         1.         1.         0.93612774]
 [0.74320242 0.94150418 1.         0.85007974 0.88622754]
 [0.96676737 0.93129062 0.86910995 1.         0.71057884]
 [0.70392749 0.71587744 0.81675393 0.67942584 1.        ]]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3866237848.py, line 1)

In [44]:
print(clfs[3].fit(X_cell[0],y_cell[0]).score(X_cell[1],y_cell[1]))

0.8319405756731661


In [37]:
model = [clfs[4].fit(X_cell[0],y_cell[0])]
print(model)

[Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(alpha=10.0, early_stopping=True,
                               hidden_layer_sizes=[100, 10], max_iter=2000,
                               random_state=1, solver='lbfgs'))])]


In [45]:
print(model[0].score(X_cell[1],y_cell[1]))


0.8263695450324977


In [7]:
print(Xlist_n1)

['/data/yangf7/identity/night1/night1_N1_FCs.csv', '/data/yangf7/identity/night1/night1_N2_FCs.csv', '/data/yangf7/identity/night1/night1_N3_FCs.csv', '/data/yangf7/identity/night1/night1_REM_FCs.csv', '/data/yangf7/identity/night1/night1_Wake_FCs.csv']
